In [ ]:
import pandas as pd

# Tables and figures of the paper, from the dataset
dataset = pd.read_csv('cltbench.csv')
EXAM_TYPES = {'public_service': 'Public-sector job exam', 'bar_exam': 'Brazilian Bar Exam'}
BOARDS = ['FCC', 'FGV', 'ESAF', 'CESPE/CEBRASPE']
print(f'{len(dataset)} questions in {dataset["exam"].nunique()} exams')

In [ ]:
def period(years):
    # '2010-2024', or the year alone when all the exams are of the same year
    first, last = years.min(), years.max()
    return str(first) if first == last else f'{first}-{last}'


# Table 3: number of questions and exams by exam board and exam type. Inside each board, the exam type with more
# questions comes first
table = dataset.groupby(['board', 'exam_type']).agg(
    questions=('id', 'size'), exams=('exam', 'nunique'), period=('year', period)).reset_index()
table['board_order'] = table['board'].map(BOARDS.index)
table = table.sort_values(['board_order', 'questions'], ascending=[True, False]).drop(columns='board_order')
table['exam_type'] = table['exam_type'].map(EXAM_TYPES)
total = pd.DataFrame([{
    'board': 'Total', 'exam_type': '', 'questions': len(dataset), 'exams': dataset['exam'].nunique(),
    'period': period(dataset['year']),
}])
table = pd.concat([table, total], ignore_index=True)
table = table.rename(columns={
    'board': 'Exam board', 'exam_type': 'Exam type', 'questions': 'Questions', 'exams': 'Exams', 'period': 'Period'})
table['Questions'] = table['Questions'].map('{:,}'.format)
table

In [ ]:
import os
import matplotlib.pyplot as plt

# Fig. 2: number of questions per exam year, by exam board. The bars are stacked from the bottom in the reverse
# order of the legend; the years without exams are left as gaps
BOARD_COLORS = {'FCC': '#2a78d6', 'FGV': '#eb6834', 'ESAF': '#1baf7a', 'CESPE/CEBRASPE': '#eda100'}
FIGURES_DIR = 'figures'

counts = dataset.pivot_table(index='year', columns='board', values='id', aggfunc='size', fill_value=0)[BOARDS]
fig, ax = plt.subplots(figsize=(10, 4.5))
bottom = pd.Series(0, index=counts.index)
for board in reversed(BOARDS):
    ax.bar(counts.index, counts[board], bottom=bottom, width=0.8, color=BOARD_COLORS[board], label=board,
           edgecolor='white', linewidth=1)
    bottom += counts[board]
for year, total in bottom.items():
    ax.text(year, total + 2, str(total), ha='center', va='bottom', fontsize=8, color='#333333')
ax.set_xticks(counts.index)
ax.set_xticklabels(counts.index, rotation=45)
ax.set_xlabel('Year')
ax.set_ylabel('Number of questions')
ax.set_ylim(0, bottom.max() * 1.1)
ax.grid(axis='y', color='#e5e5e5', linewidth=0.8)
ax.set_axisbelow(True)
for side in ['top', 'right']:
    ax.spines[side].set_visible(False)
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[::-1], labels[::-1], title='Exam Board', frameon=True, loc='upper left')
fig.tight_layout()
os.makedirs(FIGURES_DIR, exist_ok=True)
fig.savefig(f'{FIGURES_DIR}/questions_per_year_by_board.png', dpi=300)
fig.savefig(f'{FIGURES_DIR}/questions_per_year_by_board.pdf')
counts.assign(Total=bottom)

In [ ]:
# Fig. 3: number of tokens (o200k_base) of the statement and of the statement + options, by exam type
TYPE_COLORS = {'public_service': '#2a78d6', 'bar_exam': '#eb6834'}
LEGEND_NAMES = {'public_service': 'Public-sector job exams', 'bar_exam': 'Brazilian Bar Exam'}
GROUPS = [('Statement', 'num_tokens'), ('Statement + options', 'num_tokens_total')]

fig, ax = plt.subplots(figsize=(10, 4.5))
positions = {('Statement', 'public_service'): 3.8, ('Statement', 'bar_exam'): 3.2,
             ('Statement + options', 'public_service'): 1.8, ('Statement + options', 'bar_exam'): 1.2}
for (group, column) in GROUPS:
    for exam_type, color in TYPE_COLORS.items():
        values = dataset.loc[dataset['exam_type'] == exam_type, column]
        ax.boxplot(values, positions=[positions[(group, exam_type)]], vert=False, widths=0.5, patch_artist=True,
                   boxprops=dict(facecolor=color, edgecolor='#555555'), medianprops=dict(color='black', linewidth=1.5),
                   whiskerprops=dict(color='#555555'), capprops=dict(color='#555555'),
                   flierprops=dict(marker='o', markersize=3, markerfacecolor='none', markeredgecolor='#888888'))
ax.set_yticks([3.5, 1.5])
ax.set_yticklabels([group for group, _ in GROUPS])
ax.set_xlabel('Number of tokens (o200k_base)')
ax.set_xlim(left=0)
ax.grid(axis='x', color='#e5e5e5', linewidth=0.8)
ax.set_axisbelow(True)
for side in ['top', 'right']:
    ax.spines[side].set_visible(False)
counts_by_type = dataset['exam_type'].value_counts()
handles = [plt.Rectangle((0, 0), 1, 1, facecolor=color, edgecolor='#555555') for color in TYPE_COLORS.values()]
labels = [f'{LEGEND_NAMES[t]} (n = {counts_by_type[t]:,})' for t in TYPE_COLORS]
ax.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 1.12), ncol=2, frameon=False)
fig.tight_layout()
fig.savefig(f'{FIGURES_DIR}/tokens_by_exam_type.png', dpi=300)
fig.savefig(f'{FIGURES_DIR}/tokens_by_exam_type.pdf')

In [ ]:
# Numbers of the text of Fig. 3: medians of the tokens and share of questions whose options have more tokens than
# the statement
summary = dataset.groupby('exam_type')[['num_tokens', 'num_tokens_options', 'num_tokens_total']].median()
summary.loc['all'] = dataset[['num_tokens', 'num_tokens_options', 'num_tokens_total']].median()
options_longer = (dataset['num_tokens_options'] > dataset['num_tokens']).mean()
print(f'Options with more tokens than the statement: {options_longer:.1%}')
summary.rename(index=EXAM_TYPES)

In [ ]:
# Table 4: number (%) of questions by correct option, for questions with four and five options
table4 = pd.crosstab(dataset['answer'], dataset['num_options'])
for n_options in table4.columns:
    column = table4[n_options]
    table4[n_options] = [f'{count} ({count / column.sum():.1%})' if count else '-' for count in column]
table4['All questions'] = dataset['answer'].value_counts().sort_index().map('{:,}'.format)
table4.loc['Total'] = [f'{(dataset["num_options"] == n).sum():,}' for n in table4.columns[:-1]] + [f'{len(dataset):,}']
table4 = table4.rename(columns={4: '4-option questions', 5: '5-option questions'})
table4.index.name = 'Correct Option'
table4.columns.name = None
table4

In [ ]:
import json
import textwrap

# Example of a record of the dataset as a card: id, board, year and exam type; the exam; the statement; the options
# with the correct one highlighted; and the other fields
CARD_ID = 'CLTBench-0001'
CARD_WIDTH, MARGIN, LINE, WRAP = 6.4, 0.25, 0.19, 92
TEXT, MUTED, ACCENT, ACCENT_FILL, BORDER = '#222222', '#6b6b6b', '#2a78d6', '#eaf2fc', '#d6d6d6'


def card_lines(record):
    # [(kind, text), ...] of the card, kind chooses the style and the spacing
    options = json.loads(record['options'])
    lines = [('header', ''), ('exam', record['exam']), ('gap', '')]
    lines += [('text', line) for line in textwrap.wrap(record['question'], WRAP)]
    lines.append(('gap', ''))
    for letter, option in options.items():
        wrapped = textwrap.wrap(option, WRAP - 7)
        kind = 'correct' if letter == record['answer'] else 'option'
        lines += [(kind, (f'({letter})' if k == 0 else '', line)) for k, line in enumerate(wrapped)]
        lines.append(('option_gap', ''))
    lines += [('rule', ''),
              ('meta', f'position: {record["position"]}   ·   institution: {record["institution"]}'),
              ('meta', f'answer: {record["answer"]}   ·   question_number: {record["question_number"]}   ·   '
                       f'num_options: {record["num_options"]}   ·   num_words: {record["num_words"]}   ·   '
                       f'num_tokens: {record["num_tokens"]}')]
    return lines


def draw_card(record, path):
    lines = card_lines(record)
    steps = {'gap': 0.5, 'option_gap': 0.35, 'rule': 0.6}
    height = 2 * MARGIN + LINE * sum(steps.get(kind, 1) for kind, _ in lines)
    fig = plt.figure(figsize=(CARD_WIDTH, height))
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set_xlim(0, CARD_WIDTH)
    ax.set_ylim(height, 0)
    ax.axis('off')
    ax.add_patch(plt.Rectangle((0.03, 0.03), CARD_WIDTH - 0.06, height - 0.06, fill=False, edgecolor=BORDER, linewidth=1))
    y = MARGIN + LINE / 2
    correct_rows = []
    for kind, text in lines:
        if kind == 'header':
            ax.text(MARGIN, y, record['id'], fontsize=10, fontweight='bold', color=TEXT, va='center')
            ax.text(CARD_WIDTH - MARGIN, y, f'{record["board"]}  ·  {record["year"]}  ·  {record["exam_type"]}',
                    fontsize=8, color=MUTED, va='center', ha='right')
        elif kind == 'exam':
            ax.text(MARGIN, y, text, fontsize=8, color=MUTED, va='center')
        elif kind == 'text':
            ax.text(MARGIN, y, text, fontsize=8.5, color=TEXT, va='center')
        elif kind in ('option', 'correct'):
            letter, line = text
            weight = 'bold' if kind == 'correct' else 'normal'
            ax.text(MARGIN + 0.05, y, letter, fontsize=8.5, color=TEXT, va='center', fontweight=weight)
            ax.text(MARGIN + 0.5, y, line, fontsize=8.5, color=TEXT, va='center')
            if kind == 'correct':
                correct_rows.append(y)
        elif kind == 'rule':
            ax.plot([MARGIN, CARD_WIDTH - MARGIN], [y - LINE * 0.3] * 2, color=BORDER, linewidth=0.8)
        elif kind == 'meta':
            ax.text(MARGIN, y, text, fontsize=6.5, color=MUTED, va='center')
        y += LINE * steps.get(kind, 1)
    if correct_rows:
        top, bottom = correct_rows[0] - LINE * 0.75, correct_rows[-1] + LINE * 0.75
        ax.add_patch(plt.Rectangle((MARGIN - 0.05, top), CARD_WIDTH - 2 * MARGIN + 0.1, bottom - top,
                                   facecolor=ACCENT_FILL, edgecolor=ACCENT, linewidth=1, zorder=0))
    fig.savefig(f'{path}.png', dpi=300)
    fig.savefig(f'{path}.pdf')
    return fig


card = draw_card(dataset.set_index('id').loc[CARD_ID].to_dict() | {'id': CARD_ID},
                 f'{FIGURES_DIR}/example_{CARD_ID.lower()}')